# APEX Go2 を MuJoCo Warp で動かす

元 APEX の Go2 平坦地形設定を対象に、物理シミュレータを MuJoCo Warp に差し替えます。学習器は元リポジトリの Multi-Critic PPO を使用します。Linux、NVIDIA CUDA GPU、Python 3.10 以降を想定します。まず少数環境で観測と報酬を確認してから学習してください。


## 1. 依存ライブラリ

CUDA 対応 PyTorch は [公式手順](https://pytorch.org/get-started/locally/)に従って先に導入してください。以下のセルは MuJoCo Warp と学習記録用ライブラリを追加します。インストール後、必要ならカーネルを再起動します。


In [ ]:
%pip install "mujoco-warp==3.14.0" pandas pyyaml wandb tensorboard


In [ ]:
import torch
assert torch.cuda.is_available(), "CUDA 対応の PyTorch と NVIDIA GPU が必要です"
print("PyTorch", torch.__version__, "GPU", torch.cuda.get_device_name(0))


## 2. 元 APEX を取得

コード、模倣 CSV、Go2 URDF、Multi-Critic PPO は公開リポジトリの指定コミットを使います。


In [ ]:
from pathlib import Path
import subprocess, sys

PACKAGE_DIR = Path.cwd().resolve()
APEX_ROOT = PACKAGE_DIR / "APEX"
APEX_COMMIT = "f35c54a4fec00c03751e0d28187278313cde51d9"
if not APEX_ROOT.exists():
    subprocess.run(["git", "clone", "https://github.com/marmotlab/APEX.git", str(APEX_ROOT)], check=True)
subprocess.run(["git", "-C", str(APEX_ROOT), "checkout", APEX_COMMIT], check=True)
sys.path.insert(0, str(PACKAGE_DIR))
sys.path.insert(0, str(APEX_ROOT / "rsl_rl"))
print("APEX:", APEX_ROOT)


## 3. モデルと環境のスモークテスト

初回の MuJoCo Warp コンパイルには時間がかかります。観測は `(N,45)`、critic 観測は `(N,77)`、報酬は `(N,2)` のはずです。


In [ ]:
from go2_mjcf import build_go2_mjcf
import mujoco

urdf = APEX_ROOT / "resources/robots/go2/urdf/go2.urdf"
cpu_model = mujoco.MjModel.from_xml_string(build_go2_mjcf(urdf))
print("nq, nv, nu:", cpu_model.nq, cpu_model.nv, cpu_model.nu)
assert (cpu_model.nq, cpu_model.nv, cpu_model.nu) == (19, 18, 12)


In [ ]:
from env import ApexGo2Warp

env = ApexGo2Warp(APEX_ROOT, num_envs=16)
obs, critic_obs = env.reset()
for _ in range(5):
    obs, critic_obs, reward, done, info = env.step(torch.zeros(16, 12, device="cuda"))
assert obs.shape == (16, 45)
assert critic_obs.shape == (16, 77)
assert reward.shape == (16, 2)
assert torch.isfinite(obs).all() and torch.isfinite(critic_obs).all() and torch.isfinite(reward).all()
print("OK", obs.shape, critic_obs.shape, reward.shape, "mean reward", reward.mean(dim=0).tolist())


## 4. 元の Multi-Critic PPO で 1 回だけ更新

学習器は APEX 内の `rsl_rl` を変更せず利用します。W&B は既定でオフラインです。初回は 64 環境・1 更新で接続を確認します。


In [ ]:
from train import make_runner
env, runner = make_runner(APEX_ROOT, num_envs=64, log_dir=PACKAGE_DIR / "logs" / "smoke")
runner.learn(num_learning_iterations=1, init_at_random_ep_len=False)
print("checkpoint:", list((PACKAGE_DIR / "logs" / "smoke").glob("model_*.pt")))


## 5. 本学習

1 更新が通った後で環境数と更新回数を増やします。元設定は 4096 環境・1200 更新ですが、必要な GPU メモリと実行時間は実機で測ってください。以下は手動で実行するセルです。


In [ ]:
# env, runner = make_runner(APEX_ROOT, num_envs=4096, log_dir=PACKAGE_DIR / "logs" / "full")
# runner.learn(num_learning_iterations=1200, init_at_random_ep_len=False)
